# Base di conoscenza per il modulo RAG — Marine AI

Costruisce, per ciascuna delle 196 specie del classificatore, un documento testuale che combina tre fonti gia' validate nei notebook precedenti:

- **Caption scientifiche esperte** dalle immagini ORCA (notebook 01)
- **Tassonomia** risolta su WoRMS (notebook 02)
- **Trend e anomalie** dalle occorrenze OBIS (notebook 04), quando disponibili

I documenti vengono trasformati in embedding con un modello locale (nessuna chiave API richiesta) e indicizzati in ChromaDB, un vector store persistente su disco. Questo indice sara' lo strumento di recupero usato dagli agenti CrewAI nel notebook successivo.

## 0. Setup

In [7]:
# pip install chromadb sentence-transformers pandas --quiet

import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

def find_project_root(marker=".gitignore"):
    path = Path.cwd()
    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent
    raise FileNotFoundError(f"Marker '{marker}' non trovato risalendo da {Path.cwd()}")

PROJECT_ROOT = find_project_root()
ORCA_DIR = PROJECT_ROOT / "data" / "orca"
SPLIT_DIR = ORCA_DIR / "split_annotations"
DOCS_DIR = PROJECT_ROOT / "docs"
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Vector store dir:", VECTOR_STORE_DIR)

Project root: /home/ubuntu/marine_ai
Vector store dir: /home/ubuntu/marine_ai/data/vector_store


## 1. Caption scientifiche per specie (da ORCA)

In [8]:
coverage_df = pd.read_csv(DOCS_DIR / "species_environmental_coverage.csv")
LABEL_MAP = dict(zip(coverage_df["common_name"], coverage_df["canonical_label"]))

def collect_captions(split_path, label_map, max_per_species=4):
    with open(split_path) as f:
        coco = json.load(f)
    cat_by_id = {c["id"]: c["name"] for c in coco["categories"]}

    captions = defaultdict(list)
    for ann in coco["annotations"]:
        if ann["label"] != 2:  # solo caption esperte rifinite, non quelle generate da LLM (label 0)
            continue
        common_name = cat_by_id.get(ann["category_id"])
        canonical = label_map.get(common_name)
        if canonical is None:
            continue
        caption = ann.get("caption", "").strip()
        if caption and len(captions[canonical]) < max_per_species:
            captions[canonical].append(caption)
    return captions

captions_by_species = collect_captions(SPLIT_DIR / "intra_class_train.json", LABEL_MAP)
print(f"Specie con almeno una caption esperta: {len(captions_by_species)} / 196")

Specie con almeno una caption esperta: 194 / 196


## 2. Tassonomia e dati ambientali per specie

In [9]:
taxonomy_df = coverage_df.drop_duplicates(subset="canonical_label").set_index("canonical_label")

trends_path = DOCS_DIR / "species_yearly_trends.csv"
anomalies_path = DOCS_DIR / "species_anomalies.csv"
trends_df = pd.read_csv(trends_path) if trends_path.exists() else pd.DataFrame()
anomalies_df = pd.read_csv(anomalies_path) if anomalies_path.exists() else pd.DataFrame()

def environmental_summary(canonical_label):
    species_trend = trends_df[trends_df["canonical_label"] == canonical_label] if not trends_df.empty else pd.DataFrame()
    if species_trend.empty:
        return "Dati ambientali non disponibili per questa classe."
    n_years = species_trend["date_year"].nunique()
    year_min, year_max = species_trend["date_year"].min(), species_trend["date_year"].max()
    species_anomalies = anomalies_df[anomalies_df["canonical_label"] == canonical_label] if not anomalies_df.empty else pd.DataFrame()
    anomaly_years = sorted(species_anomalies["date_year"].tolist()) if not species_anomalies.empty else []
    text = f"Osservazioni campionate su OBIS disponibili dal {year_min} al {year_max} ({n_years} anni con dati)."
    if anomaly_years:
        text += f" Anni con numero di osservazioni anomalo rispetto alla media della serie: {', '.join(map(str, anomaly_years))}."
    return text

## 3. Assemblaggio del documento per specie

In [10]:
def build_species_document(canonical_label):
    row = taxonomy_df.loc[canonical_label] if canonical_label in taxonomy_df.index else None
    parts = [f"Specie: {canonical_label}."]

    if row is not None:
        taxa = [row.get("kingdom"), row.get("phylum"), row.get("class"), row.get("order"), row.get("family"), row.get("genus")]
        taxa = [t for t in taxa if isinstance(t, str) and t]
        if taxa:
            parts.append("Classificazione tassonomica: " + " > ".join(taxa) + ".")
        if isinstance(row.get("accepted_name"), str):
            parts.append(f"Nome scientifico accettato (WoRMS): {row['accepted_name']}.")
        if not bool(row.get("taxonomy_reliable", True)):
            parts.append("Nota: la tassonomia di questa classe e' segnalata come non pienamente affidabile nei metadati sorgente.")

    captions = captions_by_species.get(canonical_label, [])
    if captions:
        parts.append("Descrizioni visive da osservazioni fotografiche: " + " ".join(captions))

    parts.append(environmental_summary(canonical_label))

    return " ".join(parts)

all_species = sorted(coverage_df["canonical_label"].unique())
species_documents = {label: build_species_document(label) for label in all_species}

print(f"Documenti costruiti: {len(species_documents)}")
print("\n--- Esempio ---\n")
example_label = "green sea turtle" if "green sea turtle" in species_documents else all_species[0]
print(species_documents[example_label])

Documenti costruiti: 196

--- Esempio ---

Specie: green sea turtle. Classificazione tassonomica: Animalia > Chordata > Reptilia > Testudines > Cheloniidae > Chelonia. Nome scientifico accettato (WoRMS): Chelonia mydas. Descrizioni visive da osservazioni fotografiche: This is an image of a green sea turtle. It is a large, green sea turtle with its head up and its brown eyes open, and its belly and the undersides of its limbs are yellow and white. This image appears to be a sea turtle swimming in the ocean towards the right. It has a round, grayish-green, flat carapace with black spots on it. It has an oval-shaped head and a pair of brown eyes. The object in this figure is an image of a green sea turtle swimming in the ocean. The turtle is a green color with a distinct shell on its back and four flippers for movement. The turtule is swimming toward to surface of the water. The image shows a sea turtle swimming in the ocean. The turtle is facing towards the right side of the image and it

## 4. Embedding e indicizzazione in ChromaDB

Modello di embedding locale (`all-MiniLM-L6-v2`, via sentence-transformers): nessuna chiave API, gira su CPU in pochi secondi per 196 documenti.

In [12]:
client = chromadb.HttpClient(host="127.0.0.1", port=8001)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")

collection = client.get_or_create_collection(
    name="marine_species_knowledge",
    embedding_function=embedding_fn,
    metadata={"description": "Documenti per specie: tassonomia WoRMS, caption ORCA, trend/anomalie OBIS"},
)

labels = list(species_documents.keys())
documents = list(species_documents.values())

collection.upsert(
    ids=labels,
    documents=documents,
    metadatas=[{"canonical_label": label} for label in labels],
)

print(f"Indicizzati {collection.count()} documenti in '{collection.name}'")

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Indicizzati 196 documenti in 'marine_species_knowledge'


## 5. Test di recupero

Verifica che una domanda in linguaggio naturale recuperi le specie pertinenti — prima di collegare questo indice agli agenti CrewAI.

In [13]:
test_queries = [
    "Quali specie sono state osservate con un numero di avvistamenti insolito negli ultimi anni?",
    "Descrivimi un animale con tentacoli che vive in acque tropicali",
    "Che tipo di tartaruga marina e' presente nel dataset?",
]

for query in test_queries:
    results = collection.query(query_texts=[query], n_results=3)
    print(f"Query: {query}")
    for label, distance in zip(results["ids"][0], results["distances"][0]):
        print(f"  - {label} (distanza: {distance:.3f})")
    print()

Query: Quali specie sono state osservate con un numero di avvistamenti insolito negli ultimi anni?
  - sponge (distanza: 0.419)
  - seahorse (distanza: 0.484)
  - sea whip (distanza: 0.493)

Query: Descrivimi un animale con tentacoli che vive in acque tropicali
  - snake pipefish (distanza: 0.284)
  - snake eel (distanza: 0.289)
  - thenus orientalis (distanza: 0.299)

Query: Che tipo di tartaruga marina e' presente nel dataset?
  - leatherback sea turtle (distanza: 0.175)
  - emys orbicularis (distanza: 0.234)
  - sea turtle (distanza: 0.247)



Il vector store e' persistito su disco in `data/vector_store/` — il notebook successivo (agenti CrewAI) lo apre in sola lettura con lo stesso `PersistentClient`, senza dover ricostruire l'indice ogni volta.

In [14]:
print(client.list_collections())
print(collection.count())

[Collection(name=marine_species_knowledge)]
196
